# 07 — SHGAT Graph Visualization (post-FQDN normalization)

Visualise le graph SHGAT tel qu'il est construit par `db-sync.ts` et `initializer.ts` :
- **Capabilities** (workflow_pattern) = nœuds bleus
- **Tools** (dag_structure→tools_used) = nœuds orange
- **cap→cap** edges (capability_dependency) = arêtes rouges
- **cap→tool** edges (dag_structure) = arêtes grises

Sources DB : PostgreSQL `casys`

**2026-02-20**: Re-run post data quality fix:
- dag_structure.tools_used normalisé (209→181 tools, 1039→875 edges)
- FQDN `pml.mcp.X.Y.hash` → `X:Y` appliqué côté notebook

In [1]:
import psycopg2
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import re
import json
from collections import Counter, defaultdict

conn = psycopg2.connect(
    host="localhost", port=5432, dbname="casys",
    user="casys", password="Kx9mP2vL7nQ4wRzT"
)
cur = conn.cursor()
print("Connected to casys DB")

# FQDN normalization (matches normalizeToolId in routing-resolver.ts)
def normalize_tool_id(tool_id: str) -> str:
    """Normalize FQDN (pml.mcp.server.tool.hash or local.default.server.tool.hash) to short format (server:tool)"""
    parts = tool_id.split('.')
    if len(parts) >= 4 and parts[0] in ('pml', 'local'):
        return f"{parts[2]}:{parts[3]}"
    return tool_id

UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I)
INTERNAL_RE = re.compile(r'^(code:|loop:)(?!exec_)')

def is_valid_tool(tool_id: str) -> bool:
    """Filter out UUIDs and internal ops (code:*, loop:*)"""
    if UUID_RE.match(tool_id):
        return False
    if INTERNAL_RE.match(tool_id):
        return False
    return bool(tool_id)

Connected to casys DB


In [2]:
# Load capabilities
cur.execute("""
    SELECT pattern_id, 
           COALESCE(dag_structure->>'name', LEFT(pattern_id::text, 8)) as name,
           dag_structure->'tools_used' as tools_used,
           COALESCE(hierarchy_level, 0) as hierarchy_level,
           success_rate
    FROM workflow_pattern
    WHERE code_snippet IS NOT NULL
""")
caps = cur.fetchall()
print(f"Loaded {len(caps)} capabilities")

# Load cap→cap edges
cur.execute("""
    SELECT from_capability_id, to_capability_id, edge_type, edge_source,
           observed_count, confidence_score
    FROM capability_dependency
""")
cap_edges = cur.fetchall()
print(f"Loaded {len(cap_edges)} cap→cap edges")

Loaded 341 capabilities
Loaded 54 cap→cap edges


In [3]:
G = nx.DiGraph()

# Track normalization stats
fqdn_normalized = 0
already_short = 0

# Add capability nodes
for cap_id, name, tools_used_raw, hier_level, success_rate in caps:
    cap_node = f"cap:{str(cap_id)[:8]}"
    G.add_node(cap_node, 
               node_type="capability",
               full_id=str(cap_id),
               label=name or str(cap_id)[:8],
               hierarchy_level=hier_level,
               success_rate=float(success_rate or 0))
    
    # Parse tools_used and add cap→tool edges
    tools = []
    if tools_used_raw:
        if isinstance(tools_used_raw, list):
            tools = tools_used_raw
        elif isinstance(tools_used_raw, str):
            try:
                tools = json.loads(tools_used_raw)
            except:
                pass
    
    for tool_id in tools:
        if not tool_id or not isinstance(tool_id, str):
            continue
        # Normalize FQDN → short format
        original = tool_id
        short = normalize_tool_id(tool_id)
        if short != original:
            fqdn_normalized += 1
        else:
            already_short += 1
        
        if not is_valid_tool(short):
            continue
        
        if not G.has_node(short):
            G.add_node(short, node_type="tool", label=short)
        G.add_edge(cap_node, short, edge_type="contains", edge_source="structural")

# Add cap→cap edges
for from_id, to_id, e_type, e_source, obs_count, conf in cap_edges:
    from_node = f"cap:{str(from_id)[:8]}"
    to_node = f"cap:{str(to_id)[:8]}"
    if G.has_node(from_node) and G.has_node(to_node):
        G.add_edge(from_node, to_node, 
                   edge_type=e_type, edge_source=e_source,
                   observed_count=obs_count, confidence=float(conf or 0))

cap_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'capability']
tool_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'tool']
cap_cap_edges = [(u,v) for u,v,d in G.edges(data=True) if d.get('edge_source') != 'structural']
cap_tool_edges = [(u,v) for u,v,d in G.edges(data=True) if d.get('edge_source') == 'structural']

print(f"Graph: {G.number_of_nodes()} nodes ({len(cap_nodes)} caps, {len(tool_nodes)} tools)")
print(f"Edges: {G.number_of_edges()} total ({len(cap_cap_edges)} cap→cap, {len(cap_tool_edges)} cap→tool)")
print(f"\nNormalization: {fqdn_normalized} FQDN→short, {already_short} already short format")

# Check for remaining FQDN
fqdn_remaining = [t for t in tool_nodes if '.' in t and t.split('.')[0] in ('pml', 'local')]
if fqdn_remaining:
    print(f"WARNING: {len(fqdn_remaining)} tools still in FQDN format: {fqdn_remaining[:5]}")
else:
    print("All tools in normalized short format (server:tool)")

Graph: 521 nodes (341 caps, 180 tools)
Edges: 582 total (54 cap→cap, 528 cap→tool)

Normalization: 111 FQDN→short, 492 already short format
All tools in normalized short format (server:tool)


In [4]:
# Basic stats
tool_degree = Counter()
for tool in tool_nodes:
    tool_degree[tool] = G.in_degree(tool)

cap_out_degree = Counter()
for cap in cap_nodes:
    cap_out_degree[cap] = G.out_degree(cap)

print("=== Top 20 Most-Used Tools (by # capabilities using them) ===")
for tool, count in tool_degree.most_common(20):
    print(f"  {tool:40s} → {count} caps")

print(f"\n=== Tool Degree Distribution ===")
degrees = list(tool_degree.values())
print(f"  Min: {min(degrees)}, Max: {max(degrees)}, Mean: {np.mean(degrees):.1f}, Median: {np.median(degrees):.1f}")
print(f"  Tools used by 1 cap only: {sum(1 for d in degrees if d == 1)} / {len(degrees)}")

print(f"\n=== Cap Out-Degree Distribution (# tools per cap) ===")
out_degrees = list(cap_out_degree.values())
print(f"  Min: {min(out_degrees)}, Max: {max(out_degrees)}, Mean: {np.mean(out_degrees):.1f}, Median: {np.median(out_degrees):.1f}")
print(f"  Caps with 0 tools: {sum(1 for d in out_degrees if d == 0)}")

=== Top 20 Most-Used Tools (by # capabilities using them) ===
  std:data_person                          → 21 caps
  std:psql_query                           → 18 caps
  std:data_company                         → 16 caps
  filesystem:read_file                     → 12 caps
  syson:syson_element_children             → 12 caps
  std:crypto_uuid                          → 12 caps
  erpnext:erpnext_doc_create               → 10 caps
  std:data_address                         → 9 caps
  std:crypto_hash                          → 8 caps
  playwright:browser_navigate              → 8 caps
  erpnext:erpnext_sales_order_list         → 7 caps
  erpnext:erpnext_doc_submit               → 7 caps
  playwright:browser_wait_for              → 7 caps
  erpnext:erpnext_item_list                → 7 caps
  plm:plm_bom_flatten                      → 7 caps
  erpnext:erpnext_customer_list            → 7 caps
  playwright:browser_take_screenshot       → 7 caps
  std:git_status                           → 7 

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.15, rc={
    'figure.facecolor': '#ffffff', 'axes.facecolor': '#faf1f6',
    'axes.edgecolor': '#cfc3cd', 'axes.labelcolor': '#1e1a1e',
    'text.color': '#1e1a1e', 'xtick.color': '#4d444c', 'ytick.color': '#4d444c',
    'grid.color': '#e8dfe6', 'grid.alpha': 0.6, 'grid.linewidth': 0.5,
    'savefig.facecolor': '#ffffff', 'savefig.dpi': 200, 'figure.dpi': 150,
    'axes.spines.top': False, 'axes.spines.right': False,
})
PRIMARY='#83468f'; TEAL='#4ECDC4'; WARM='#82524c'; MUTED='#988d97'; BLUE='#60a5fa'; GREEN='#4ade80'

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1 — Tool in-degree distribution
ax = axes[0]
med_deg = np.median(degrees)
sns.histplot(degrees, bins=30, ax=ax, color=PRIMARY, alpha=0.75, kde=True,
             line_kws={'linewidth': 2, 'color': WARM})
sns.rugplot(degrees, ax=ax, color=PRIMARY, alpha=0.3, height=0.04)
ax.axvline(med_deg, color=WARM, linestyle='--', linewidth=1.5)
ax.text(med_deg + 0.15, ax.get_ylim()[1] * 0.92,
        f'median={med_deg:.0f}', color=WARM, fontsize=10, va='top')
ax.set_xlabel('In-degree (# parent nodes using this leaf)')
ax.set_ylabel('# Tools')
sns.despine(ax=ax)

# Panel 2 — Cap out-degree distribution
ax = axes[1]
med_out = np.median(out_degrees)
sns.histplot(out_degrees, bins=30, ax=ax, color=TEAL, alpha=0.75, kde=True,
             line_kws={'linewidth': 2, 'color': BLUE})
sns.rugplot(out_degrees, ax=ax, color=TEAL, alpha=0.3, height=0.04)
ax.axvline(med_out, color=BLUE, linestyle='--', linewidth=1.5)
ax.text(med_out + 0.15, ax.get_ylim()[1] * 0.92,
        f'median={med_out:.0f}', color=BLUE, fontsize=10, va='top')
ax.set_xlabel('Out-degree (# leaf nodes per parent)')
ax.set_ylabel('# Capabilities')
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('07-degree-distributions.png', bbox_inches='tight')
plt.show()

In [6]:
# Connected components analysis
G_undirected = G.to_undirected()
components = list(nx.connected_components(G_undirected))
comp_sizes = sorted([len(c) for c in components], reverse=True)

print(f"Connected components: {len(components)}")
print(f"Largest: {comp_sizes[0]} nodes")
if len(comp_sizes) > 1:
    print(f"2nd largest: {comp_sizes[1]} nodes")
print(f"Isolated nodes: {sum(1 for s in comp_sizes if s == 1)}")
print(f"\nComponent size distribution (top 10):")
for i, s in enumerate(comp_sizes[:10]):
    print(f"  Component {i}: {s} nodes")

Connected components: 78
Largest: 240 nodes
2nd largest: 73 nodes
Isolated nodes: 17

Component size distribution (top 10):
  Component 0: 240 nodes
  Component 1: 73 nodes
  Component 2: 16 nodes
  Component 3: 12 nodes
  Component 4: 7 nodes
  Component 5: 7 nodes
  Component 6: 6 nodes
  Component 7: 6 nodes
  Component 8: 6 nodes
  Component 9: 5 nodes


In [7]:
# Tool co-occurrence: which tools appear together most often?
from itertools import combinations

cooccurrence = Counter()
for cap in cap_nodes:
    neighbors = [n for n in G.successors(cap) if G.nodes[n].get('node_type') == 'tool']
    for t1, t2 in combinations(sorted(neighbors), 2):
        cooccurrence[(t1, t2)] += 1

print("=== Top 20 Tool Co-occurrences ===")
for (t1, t2), count in cooccurrence.most_common(20):
    print(f"  {t1:30s} + {t2:30s} → {count} caps")

=== Top 20 Tool Co-occurrences ===
  std:data_company               + std:data_person                → 13 caps
  std:data_address               + std:data_person                → 7 caps
  std:data_address               + std:data_company               → 6 caps
  std:crypto_uuid                + std:data_person                → 6 caps
  syson:syson_diagram_arrange    + syson:syson_diagram_snapshot   → 5 caps
  playwright:browser_evaluate    + playwright:browser_wait_for    → 5 caps
  syson:syson_diagram_arrange    + syson:syson_diagram_drop       → 5 caps
  std:crypto_uuid                + std:data_company               → 5 caps
  playwright:browser_navigate    + playwright:browser_take_screenshot → 5 caps
  std:git_log                    + std:psql_query                 → 5 caps
  erpnext:erpnext_doc_create     + erpnext:erpnext_doc_submit     → 4 caps
  plm:plm_bom_cost               + plm:plm_bom_flatten            → 4 caps
  syson:syson_diagram_drop       + syson:syson_diagram_snaps

In [8]:
# Hierarchy level distribution
hier_levels = [G.nodes[n].get('hierarchy_level', 0) for n in cap_nodes]
level_counts = Counter(hier_levels)

print("=== Hierarchy Level Distribution ===")
for level in sorted(level_counts.keys()):
    print(f"  L{level}: {level_counts[level]} capabilities")

# Cap→cap edges analysis  
print(f"\n=== Cap→Cap Edges ({len(cap_cap_edges)} total) ===")
edge_types = Counter()
for u, v, d in G.edges(data=True):
    if d.get('edge_source') != 'structural':
        edge_types[d.get('edge_type', 'unknown')] += 1
for et, cnt in edge_types.most_common():
    print(f"  {et}: {cnt}")

=== Hierarchy Level Distribution ===
  L0: 3 capabilities
  L1: 326 capabilities
  L2: 12 capabilities

=== Cap→Cap Edges (54 total) ===
  contains: 54


In [9]:
# Interactive visualization with pyvis (for the largest component)
from pyvis.network import Network

# Get largest component
largest_comp = max(components, key=len)
subG = G.subgraph(largest_comp).copy()

# If too large, sample: keep top-degree tools + their connected caps
MAX_NODES = 200
if subG.number_of_nodes() > MAX_NODES:
    sub_tools = [n for n in subG.nodes() if subG.nodes[n].get('node_type') == 'tool']
    sub_tools_sorted = sorted(sub_tools, key=lambda t: subG.in_degree(t), reverse=True)
    keep_tools = set(sub_tools_sorted[:40])
    keep_nodes = set(keep_tools)
    for tool in keep_tools:
        for pred in subG.predecessors(tool):
            keep_nodes.add(pred)
    for u, v in cap_cap_edges:
        if u in keep_nodes or v in keep_nodes:
            keep_nodes.add(u)
            keep_nodes.add(v)
    subG = subG.subgraph(keep_nodes).copy()
    print(f"Sampled to {subG.number_of_nodes()} nodes (top-40 tools + neighbors)")

net = Network(height='800px', width='100%', directed=True, 
              bgcolor='#08080a', font_color='white',
              notebook=True, cdn_resources='remote')
net.barnes_hut(gravity=-3000, spring_length=150)

for node, data in subG.nodes(data=True):
    if data.get('node_type') == 'capability':
        label = data.get('label', node)[:20]
        level = data.get('hierarchy_level', 0)
        colors = {0: '#4A90D9', 1: '#6FA8FF', 2: '#A0C4FF'}
        net.add_node(node, label=label, color=colors.get(level, '#A0C4FF'),
                     size=12 + level * 5, title=f"{node}\nL{level}")
    else:
        deg = subG.in_degree(node)
        net.add_node(node, label=node, color='#FFB86F',
                     size=8 + min(deg * 2, 20), 
                     shape='diamond',
                     title=f"{node}\nUsed by {deg} caps")

for u, v, data in subG.edges(data=True):
    if data.get('edge_source') == 'structural':
        net.add_edge(u, v, color='rgba(255,184,111,0.3)', width=1)
    else:
        net.add_edge(u, v, color='#FF6B6B', width=3,
                     title=f"{data.get('edge_type','?')} ({data.get('edge_source','?')})")

net.show('07-shgat-graph.html')
print(f"Saved interactive graph: 07-shgat-graph.html")

Sampled to 189 nodes (top-40 tools + neighbors)
07-shgat-graph.html
Saved interactive graph: 07-shgat-graph.html


In [10]:
# Hub tools analysis: tools that act as "bridges" between many capabilities
print("=== Hub Tools (potential MP clustering culprits) ===")
print("Tools connected to 10+ capabilities — MP spreads through these:\n")

hub_tools = [(t, G.in_degree(t)) for t in tool_nodes if G.in_degree(t) >= 10]
hub_tools.sort(key=lambda x: -x[1])

for tool, deg in hub_tools:
    pct = 100 * deg / len(cap_nodes)
    print(f"  {tool:40s} {deg:3d} caps ({pct:4.1f}%)")

print(f"\n{len(hub_tools)} hub tools (>= 10 caps)")
print(f"These connect {sum(d for _,d in hub_tools)} cap→tool edges")
print(f"out of {len(cap_tool_edges)} total = {100*sum(d for _,d in hub_tools)/max(len(cap_tool_edges),1):.1f}%")
print("\n→ If MP weight is too high, signal leaks through hub tools")
print("  making all capabilities connected to e.g. 'std:psql_query' look similar.")

=== Hub Tools (potential MP clustering culprits) ===
Tools connected to 10+ capabilities — MP spreads through these:

  std:data_person                           21 caps ( 6.2%)
  std:psql_query                            18 caps ( 5.3%)
  std:data_company                          16 caps ( 4.7%)
  filesystem:read_file                      12 caps ( 3.5%)
  syson:syson_element_children              12 caps ( 3.5%)
  std:crypto_uuid                           12 caps ( 3.5%)
  erpnext:erpnext_doc_create                10 caps ( 2.9%)

7 hub tools (>= 10 caps)
These connect 101 cap→tool edges
out of 528 total = 19.1%

→ If MP weight is too high, signal leaks through hub tools
  making all capabilities connected to e.g. 'std:psql_query' look similar.


In [11]:
# Server namespace breakdown
server_tools = defaultdict(list)
for tool in tool_nodes:
    server = tool.split(':')[0] if ':' in tool else 'unknown'
    server_tools[server].append(tool)

print("=== Tools by Server Namespace ===")
for server, tools in sorted(server_tools.items(), key=lambda x: -len(x[1])):
    total_deg = sum(G.in_degree(t) for t in tools)
    print(f"  {server:20s} {len(tools):3d} tools  {total_deg:4d} cap→tool edges")

=== Tools by Server Namespace ===
  std                   49 tools   166 cap→tool edges
  erpnext               34 tools   111 cap→tool edges
  syson                 18 tools    65 cap→tool edges
  code                  14 tools    14 cap→tool edges
  filesystem            12 tools    38 cap→tool edges
  plm                   11 tools    36 cap→tool edges
  fake                  10 tools    22 cap→tool edges
  playwright             9 tools    40 cap→tool edges
  onshape                7 tools    13 cap→tool edges
  sim                    5 tools     8 cap→tool edges
  db                     3 tools     4 cap→tool edges
  memory                 2 tools     4 cap→tool edges
  meta                   2 tools     2 cap→tool edges
  faker                  1 tools     1 cap→tool edges
  exa                    1 tools     2 cap→tool edges
  color                  1 tools     1 cap→tool edges
  fetch                  1 tools     1 cap→tool edges


In [12]:
# Cleanup
cur.close()
conn.close()
print("Done.")

Done.
